# Preprocessing target data for use with Abil.py
Target (the variable you want to model) data needs to be put on the same coordinate grid as your predictor (environmental) data.
You'll need to load your observations, do any additional data cleaning (dropping certain methods, any out of range data), and then convert the coordinates to match the predictor data.

### Import required packages

In [1]:
import pandas as pd
import xarray as xr
import numpy as np
from abil.pseudo_generation import generate_pseudo_absences

### Define paths

In [2]:

from pathlib import Path

def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start).resolve()
    for path in [start, *start.parents]:
        if (path / "environment.yml").exists():
            return path
    raise FileNotFoundError("Could not find project root containing environment.yml")


PROJECT_ROOT = find_project_root()


### Define your observation data filename and variable of interest

In [3]:
filename = PROJECT_ROOT / "data" / "global-prochlorococcus-synechococcus-and-picoeukaryotic-phytoplankton_v2.csv"
varname = ['Prochlorococcus']


### Load data
I'm using usecols here to specifically extract the columns I need, as this is a large spreadsheet and contains a lot of extra information that I'd just drop later

In [4]:
d_raw = pd.read_csv(filename,delimiter='\t',na_values='nd',usecols=['Year','Latitude','Longitude','Depth','Day','Prochlorococcus'])

In [5]:
mask = d_raw['Year'].notna() & d_raw['Day'].notna()

d_raw['Month'] = np.nan

d_raw.loc[mask, 'Month'] = pd.to_datetime(
    d_raw.loc[mask, 'Year'].astype(int).astype(str) +
    d_raw.loc[mask, 'Day'].astype(int).astype(str).str.zfill(3),
    format='%Y%j'
).dt.month

d_raw['Longitude'] = ((d_raw['Longitude'] + 180) % 360) - 180

In [6]:
d = d_raw.rename(columns={
    "Latitude":"lat", 
    "Longitude":"lon", 
    "Depth":"depth", 
    "Month":"time"})

# Subset dataframe to only necessary variable
d = d[['lat','lon','depth','time','Prochlorococcus']]

# Drop any rows with missing values
d.dropna(inplace=True)

# Bin data to match grid of predictor data (180x360x41x12)
# Grid to 180×360×41×12
depth_bins = np.linspace(0, 205, 42)
depth_labels = np.linspace(0, 200, 41)
d["depth"] = pd.cut(d["depth"], depth_bins, labels=depth_labels, include_lowest=True).astype(float)

lat_bins = np.linspace(-90, 90, 181)
lat_labels = np.linspace(-90, 89, 180)
d["lat"] = pd.cut(d["lat"], lat_bins, labels=lat_labels).astype(float)

lon_bins = np.linspace(-180, 180, 361)
lon_labels = np.linspace(-180, 179, 360)
d["lon"] = pd.cut(d["lon"], lon_bins, labels=lon_labels).astype(float)


# Group data so there is only one observation per gridcell
d = d.groupby(["lat", "lon", "depth", "time"]).mean().reset_index()
d.set_index(["lat", "lon", "depth", "time"], inplace=True)



### Load environmental predictor data

In [7]:
# Load environment data
ds_env = xr.open_dataset(PROJECT_ROOT / "data" / "env_data.nc")
df_env = ds_env.to_dataframe().reset_index()
env_cols = ["temperature", "no3", "o2", "PAR"]
df_env = df_env[env_cols + ["lat", "lon", "depth", "time"]].set_index(["lat", "lon", "depth", "time"])
print(f"Environment rows: {len(df_env)}")

# Filter only rows whose environmental data exists AND is fully non-NaN
valid_env_idx = df_env.dropna().index
d = d.loc[d.index.intersection(valid_env_idx)]
print(f"Rows after dropping samples without complete environmental data: {len(d)}")

# Merge with environment data
d = d.join(df_env, how="inner").reset_index()
d = d.set_index(["lat", "lon", "depth", "time"])
print(f"Rows after merging with environment: {len(d)}")


Environment rows: 31881600
Rows after dropping samples without complete environmental data: 17306
Rows after merging with environment: 17306


In [8]:
missing_df_env = df_env[~df_env.index.isin(d.index)]

df = generate_pseudo_absences(d, missing_df_env, env_cols, ['Prochlorococcus'], min_presence=1)
print("Non-unique entries found:", df.index.duplicated().sum())

Non-unique entries found: 0


### Merge observations, pseudo zeros, and environmental predictors

In [9]:
# Save combined CSV
outfile = PROJECT_ROOT / "data" / "training_pro.csv"
outfile.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(outfile, index=False)

print("\n=== Finished ===")
print("Saved combined CSV to:")
print(outfile)


=== Finished ===
Saved combined CSV to:
/home/mv23682/Documents/Abil_tutorial/data/training_pro.csv
